# Mutual Fund Recommender (mfapi.in)

This notebook builds a mutual-fund research and recommendation engine using **only [mfapi.in](https://www.mfapi.in/)**.

It does five things:

1. Downloads the Indian mutual-fund universe and caches every NAV history locally.
2. Keeps five equity categories: **large cap, mid cap, small cap, flexi cap, and broad-market index funds**.
3. Computes return, risk, and consistency metrics for each fund.
4. Scores funds **inside their own category** and selects a risk-neutral top 200.
5. Builds a portfolio from your **age, horizon, SIP/lumpsum choice, and risk tolerance**.

> **Not investment advice.** mfapi.in does not provide AUM, expense ratios, fund-manager data, or holdings. Returns are historical and do not predict the future. The "top 200" is a computed quality ranking, not a size ranking.

## 1. Setup

Everything is cached in `data/cache/`. The first run downloads about 5,000 scheme histories; later runs reuse the cache.

In [1]:
from pathlib import Path
import gzip, json, re, time, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from scipy.optimize import brentq, minimize
from scipy.stats import spearmanr
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ---- settings you may want to change -------------------------------------
CACHE = Path("data/cache")   # all downloads are cached here
TOP_N = 200                  # size of the candidate pool
MIN_YEARS = 1.5              # minimum NAV history
MIN_OBS = 300                # minimum daily NAV observations
RISK_FREE = 0.065            # annual risk-free rate (assumption)
REFRESH_DAYS = 2             # re-download the scheme list after N days
CATEGORIES = ["large", "mid", "small", "flexi", "index"]
BASE_QUOTA = 40              # equal starting quota per category
# --------------------------------------------------------------------------

RAW = CACHE / "raw"
CACHE.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)
(CACHE.parent / ".gitignore").write_text("*\n")   # keep cache out of git
print("Cache:", CACHE.resolve())

Cache: /home/hsrkl/Projects/mflyzer/data/cache


### Small helpers

- `get_json` retries network calls.
- `fetch_scheme` caches each raw response as gzip, so re-runs are fast.
- `xirr` and `sip_xirr` measure money-weighted returns for SIPs.

In [2]:
def get_json(url, tries=4):
    '''GET a URL and return JSON, retrying on temporary failures.'''
    for attempt in range(tries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(1 + attempt)

def fresh(path, days=REFRESH_DAYS):
    '''True when a cache file exists and is recent.'''
    return path.exists() and (time.time() - path.stat().st_mtime) < days * 86400

def fetch_scheme(code):
    '''Return one scheme's JSON. Raw responses are cached as gzip.'''
    path = RAW / f"{code}.json.gz"
    if path.exists():
        with gzip.open(path, "rt") as file:
            return json.load(file)
    data = get_json(f"https://api.mfapi.in/mf/{code}")
    with gzip.open(path, "wt") as file:
        json.dump(data, file)
    return data

def xirr(flows):
    '''Annualised money-weighted return for [(date, cashflow), ...].'''
    if len(flows) < 2:
        return np.nan
    start = flows[0][0]
    years = np.array([(d - start).days / 365.25 for d, _ in flows])
    amounts = np.array([v for _, v in flows])
    try:
        return brentq(lambda rate: np.sum(amounts / (1 + rate) ** years), -0.999, 5)
    except ValueError:
        return np.nan

def sip_xirr(nav):
    '''XIRR of a monthly SIP of 1 rupee into a NAV series.'''
    nav = nav.dropna()
    nav = nav[nav.index >= nav.index.max() - pd.DateOffset(months=60)]
    if len(nav) < 12:
        return np.nan
    units, flows = 0.0, []
    for month in pd.date_range(nav.index.min(), nav.index.max(), freq="MS"):
        available = nav.index[nav.index >= month]
        if len(available) == 0:
            continue
        day = available[0]
        units += 1 / nav.loc[day]
        flows.append((day, -1.0))
    flows.append((nav.index[-1], units * nav.iloc[-1]))
    return xirr(flows)

## 2. Download the fund universe

`GET /mf` returns every scheme. We keep **Direct Plan – Growth** options, then download each scheme's header and NAV history.

In [3]:
master_path = CACHE / "master.parquet"
if fresh(master_path):
    master = pd.read_parquet(master_path)
else:
    master = pd.DataFrame(get_json("https://api.mfapi.in/mf"))
    master.to_parquet(master_path, index=False)

name = master["schemeName"].str.lower()
candidates = master[
    name.str.contains(r"\bdirect\b")
    & name.str.contains("growth")
    & ~name.str.contains(r"idcw|dividend|bonus|payout|reinvest")
].copy()
print(f"{len(master):,} schemes total")
print(f"{len(candidates):,} Direct-Growth candidates")

def fetch_meta(code):
    data = fetch_scheme(code)
    meta = data["meta"]
    return {
        "code": code,
        "name": meta["scheme_name"],
        "amc": meta["fund_house"],
        "category_raw": meta.get("scheme_category"),
    }

with ThreadPoolExecutor(max_workers=24) as pool:
    meta = pd.DataFrame(pool.map(fetch_meta, candidates["schemeCode"]))
print(f"Downloaded/cached {len(meta):,} scheme headers")

37,982 schemes total
5,041 Direct-Growth candidates


Downloaded/cached 5,041 scheme headers


## 3. Classify and clean

mfapi's `scheme_category` field is inconsistent (values like `Income`, `IDF`, or `1099 Days` appear). So classification uses the **fund name first**, then the category text as a fallback.

We keep only broad-market index funds and the four active categories. Sectoral/thematic index funds, ETFs, FOFs, closed-ended plans, and fixed-maturity plans are excluded.

In [4]:
BROAD_INDEX = r"nifty\s*(50\b|100\b|next\s*50|500)|sensex|bse\s*(500|sensex)|midcap\s*150|smallcap\s*250|nifty\s*(large|mid|small)\s*cap|total\s*market"
SECTOR_INDEX = r"bank\b|it\b|pharma|auto\b|fmcg|consumption|infra|manufactur|energy|psu|defence|healthcare|tech|metal|realty|esg|momentum|quality|value|dividend|low\s*vol|alpha|equal\s*weight|sector|thematic|housing|insurance|financial|services"
NOT_A_FUND = r"\betf\b|exchange traded|fund of funds|\bfof\b|fmp|fixed maturity|closed?\s*ended|interval|1099|regular plan"
INTERNATIONAL = r"\bus\b|u\.s\.|overseas|global|world|international|china|japan|europe|emerging|asia|nasdaq|s&p|dow|ftse|hang seng|greater china|america|brazil|russia|canada|australia"

def classify(name, raw_category):
    '''Map a scheme to large/mid/small/flexi/index, or None.'''
    text = f"{name} {raw_category}".lower()
    if re.search(NOT_A_FUND, text) or re.search(INTERNATIONAL, text):
        return None
    if "index" in text or re.search(r"nifty|sensex|bse\s*\d", text):
        return "index" if re.search(BROAD_INDEX, text) and not re.search(SECTOR_INDEX, text) else None
    if re.search(r"large\s*(&|and)\s*mid", text):
        return None
    if re.search(r"small\s*cap|smallcap", text):
        return "small"
    if re.search(r"mid\s*cap|midcap", text):
        return "mid"
    if re.search(r"flexi\s*cap|flexicap", text):
        return "flexi"
    if re.search(r"large\s*cap|largecap|bluechip|frontline|top\s*100", text):
        return "large"
    raw = str(raw_category).lower()
    if "small cap fund" in raw: return "small"
    if "mid cap fund" in raw: return "mid"
    if "flexi cap fund" in raw: return "flexi"
    if "large cap fund" in raw: return "large"
    return None

meta["category"] = [classify(n, c) for n, c in zip(meta["name"], meta["category_raw"])]
funds = meta[meta["category"].notna()].copy()
print("Funds in the five categories:")
print(funds["category"].value_counts())

nav_rows = []
for code in funds["code"]:
    data = fetch_scheme(code)
    if not data["data"]:
        continue
    frame = pd.DataFrame(data["data"])
    frame["code"] = code
    nav_rows.append(frame)

navs = pd.concat(nav_rows, ignore_index=True)
navs["date"] = pd.to_datetime(navs["date"], format="%d-%m-%Y")
navs["nav"] = pd.to_numeric(navs["nav"], errors="coerce")
navs = navs.dropna(subset=["nav"])
navs = navs[navs["nav"] > 0]
navs = navs.drop_duplicates(["code", "date"]).sort_values(["code", "date"])
print(f"{len(navs):,} NAV rows for {navs['code'].nunique()} funds")

Funds in the five categories:
category
index    107
large     55
flexi     50
small     46
mid       44
Name: count, dtype: int64


487,677 NAV rows for 302 funds


### Quality filter

A fund stays only if it has:

- a NAV within the last 10 days (still active),
- at least **1.5 years** of history,
- at least **300 daily observations**.

We then build a monthly NAV matrix. Monthly returns are used for most metrics; daily NAVs are used for maximum drawdown.

In [5]:
nav_wide = navs.pivot_table(index="date", columns="code", values="nav").sort_index()
monthly_nav = nav_wide.resample("ME").last()
returns = monthly_nav.pct_change(fill_method=None)

stats = navs.groupby("code").agg(
    first=("date", "min"), last=("date", "max"), n_obs=("nav", "size")
)
stats["years"] = (stats["last"] - stats["first"]).dt.days / 365.25
latest = stats["last"].max()
keep = stats[
    (stats["last"] >= latest - pd.Timedelta(days=10))
    & (stats["years"] >= MIN_YEARS)
    & (stats["n_obs"] >= MIN_OBS)
].index

funds = funds[funds["code"].isin(keep)].merge(stats.reset_index(), on="code")
returns = returns[funds["code"]]
monthly_nav = monthly_nav[funds["code"]]
nav_wide = nav_wide[funds["code"]]
print(f"{len(funds)} funds pass quality checks")
print(funds["category"].value_counts())

212 funds pass quality checks
category
index    80
flexi    39
large    32
mid      31
small    30
Name: count, dtype: int64


### Benchmarks from mfapi only

There is no index feed in this build, so each benchmark is an **equal-weight composite of the matching index funds** already downloaded. This is a fund-based proxy, not the official index TRI.

In [6]:
def index_codes(pattern, exclude=None):
    return [
        row["code"] for _, row in funds.iterrows()
        if re.search(pattern, row["name"], re.I)
        and not (exclude and re.search(exclude, row["name"], re.I))
    ]

def index_composite(pattern, exclude=None):
    codes = index_codes(pattern, exclude)
    return returns[codes].mean(axis=1) if codes else pd.Series(np.nan, index=returns.index)

nifty50 = index_composite(r"nifty\s*50\b", r"next\s*50")
nifty500 = index_composite(r"nifty\s*500")
midcap150 = index_composite(r"midcap\s*150")
smallcap250 = index_composite(r"smallcap\s*250")
market = nifty500.fillna(0.5 * nifty50 + 0.3 * midcap150 + 0.2 * smallcap250).fillna(nifty50)

print("Index funds per benchmark:")
print(pd.Series({
    "Nifty 50": len(index_codes(r"nifty\s*50\b", r"next\s*50")),
    "Nifty 500": len(index_codes(r"nifty\s*500")),
    "Midcap 150": len(index_codes(r"midcap\s*150")),
    "Smallcap 250": len(index_codes(r"smallcap\s*250")),
}))

Index funds per benchmark:
Nifty 50        19
Nifty 500        6
Midcap 150      10
Smallcap 250     9
dtype: int64


## 4. Metrics

For every fund we compute:

- **Return:** 1y, 3y, 5y and since-inception CAGR, SIP XIRR
- **Risk:** annualised volatility, downside deviation, maximum drawdown
- **Risk-adjusted:** Sharpe, Sortino, Calmar
- **Consistency:** share of positive months, share of positive rolling 1-year windows
- **Benchmark-relative:** beta and alpha versus the Nifty 500 composite
- **Momentum:** trailing 12-month return

In [7]:
daily_dd = (nav_wide / nav_wide.cummax() - 1).min()

def fund_metrics(code, monthly_returns, nav_series, benchmark):
    r = monthly_returns.dropna()
    n = len(r)
    if n < 12:
        return None

    def cagr(x):
        return (1 + x).prod() ** (12 / len(x)) - 1

    r3 = r.iloc[-36:]
    r12 = r.iloc[-12:]
    c3 = cagr(r3)
    c5 = cagr(r.iloc[-60:]) if n >= 60 else np.nan
    vol = r3.std() * np.sqrt(12)
    downside = r3[r3 < 0].std() * np.sqrt(12)
    rolling12 = (1 + r).rolling(12).apply(np.prod, raw=True).dropna() - 1

    aligned = benchmark.reindex(r3.index).dropna()
    beta = np.cov(r3.loc[aligned.index], aligned)[0, 1] / np.var(aligned) if len(aligned) > 12 else np.nan
    market_cagr = cagr(aligned) if len(aligned) > 12 else np.nan
    alpha = c3 - (RISK_FREE + beta * (market_cagr - RISK_FREE)) if pd.notna(beta) else np.nan
    mdd = daily_dd.get(code, np.nan)

    return {
        "code": code,
        "cagr_1y": cagr(r12),
        "cagr_3y": c3,
        "cagr_5y": c5,
        "vol_3y": vol,
        "sharpe_3y": (c3 - RISK_FREE) / vol if vol > 0 else np.nan,
        "sortino_3y": (c3 - RISK_FREE) / downside if downside > 0 else np.nan,
        "max_drawdown": mdd,
        "calmar_3y": c3 / abs(mdd) if pd.notna(mdd) and mdd < 0 else np.nan,
        "positive_months": (r3 > 0).mean(),
        "consistency": (rolling12 > 0).mean() if len(rolling12) else np.nan,
        "sip_xirr": sip_xirr(nav_series),
        "momentum_12m": (1 + r12).prod() - 1,
        "beta": beta,
        "alpha": alpha,
    }

rows = [fund_metrics(code, returns[code], monthly_nav[code], market) for code in funds["code"]]
metrics = funds.merge(pd.DataFrame([row for row in rows if row]), on="code")
print(f"{len(metrics)} funds with metrics")

212 funds with metrics


In [8]:
metrics[["cagr_3y", "vol_3y", "sharpe_3y", "max_drawdown", "sip_xirr"]].describe().round(3)

,cagr_3y,vol_3y,sharpe_3y,max_drawdown,sip_xirr
count,212.000,212.000,212.000,212.000,212.000
mean,0.114,0.171,0.266,-0.295,0.085
std,0.056,0.028,0.321,0.102,0.057
min,-0.088,0.099,-1.546,-0.653,-0.112
25%,0.080,0.146,0.093,-0.372,0.049
50%,0.114,0.170,0.299,-0.301,0.088
75%,0.154,0.194,0.489,-0.202,0.123
max,0.235,0.237,0.829,-0.115,0.252


## 5. Score and select the top 200

Each metric is converted to a **percentile inside its own category**. That keeps the comparison fair: a small-cap fund is judged against other small-cap funds, not against a large-cap fund.

The composite is **risk-neutral**: 40% return, 30% risk-adjusted return, 20% consistency, 10% momentum. There is no bonus for being safe or risky.

The top 200 starts with 40 slots per category, then fills any unused slots with the best remaining funds.

In [9]:
WEIGHTS = {
    "cagr_3y": 0.15,
    "cagr_5y": 0.10,
    "sip_xirr": 0.15,
    "sharpe_3y": 0.12,
    "sortino_3y": 0.10,
    "calmar_3y": 0.08,
    "consistency": 0.12,
    "max_drawdown": 0.08,
    "momentum_12m": 0.10,
}
weights = pd.Series(WEIGHTS)

# Percentile rank within each category. Higher is always better.
percentiles = metrics.groupby("category")[list(WEIGHTS)].rank(pct=True)
metrics["score"] = (
    (percentiles * weights).sum(axis=1)
    / (percentiles.notna() * weights).sum(axis=1)
    * 100
)
metrics["n_metrics"] = percentiles.notna().sum(axis=1)
metrics = metrics.sort_values("score", ascending=False).reset_index(drop=True)

metrics[["name", "category", "score", "cagr_3y", "sharpe_3y", "max_drawdown"]].head(10)

,name,category,score,cagr_3y,sharpe_3y,max_drawdown
0,WhiteOak Capital Mid Cap Fund - Direct Plan - ...,mid,96.703,0.208,0.817,-0.193
1,BANK OF INDIA FLEXI CAP FUND - Direct Plan - G...,flexi,92.154,0.190,0.668,-0.237
2,ITI Flexi Cap Fund - Direct Plan - Growth Option,flexi,90.712,0.184,0.683,-0.214
3,HSBC Midcap Fund - Direct Plan - Growth,mid,90.108,0.219,0.744,-0.260
4,BANDHAN Small Cap Fund - Direct Plan - Growth,small,88.845,0.235,0.829,-0.243
5,Invesco India Large Cap Fund - Direct Plan - G...,large,88.197,0.139,0.455,-0.369
6,WhiteOak Capital Large Cap Fund - Direct Plan ...,large,86.979,0.131,0.465,-0.152
7,360 ONE Flexicap Fund - Direct Plan - GROWTH O...,flexi,86.040,0.176,0.706,-0.181
8,Invesco India Mid Cap Fund - Direct Plan - Growth,mid,85.871,0.222,0.780,-0.341
9,Taurus Large Cap Fund - Direct Plan - Growth,large,85.817,0.144,0.533,-0.351


### A small, gated ML check

Before locking the score, we test whether a simple model can predict **next-year, category-relative performance** from trailing metrics.

- Training rows are year-end snapshots from 2020 onward.
- Features are known at the snapshot date; the target is the following 12-month return rank.
- The last year-end is held out. A Ridge model is blended in **only if** out-of-sample rank IC is above 0.05.
- If the gate fails, the notebook says so and keeps the rule-based score. That is the expected outcome with this much noise.

This is deliberately modest: no deep learning, no look-ahead, and no silent overfitting.

In [10]:
def features_at(asof):
    '''Features known at 'asof', plus the next 12-month return as target.'''
    history = returns.loc[:asof]
    rows = []
    for code in history.columns:
        r = history[code].dropna()
        if len(r) < 24:
            continue
        r3 = r.iloc[-36:]
        c3 = (1 + r3).prod() ** (12 / len(r3)) - 1
        vol = r3.std() * np.sqrt(12)
        curve = (1 + r3).cumprod()
        rolling12 = (1 + r).rolling(12).apply(np.prod, raw=True).dropna() - 1
        rows.append({
            "code": code,
            "cagr": c3,
            "vol": vol,
            "sharpe": (c3 - RISK_FREE) / vol if vol > 0 else np.nan,
            "maxdd": (curve / curve.cummax() - 1).min(),
            "consistency": (rolling12 > 0).mean() if len(rolling12) else np.nan,
            "momentum": (1 + r.iloc[-6:]).prod() - 1,
            "age": len(r) / 12,
        })
    frame = pd.DataFrame(rows).set_index("code")
    frame["category"] = funds.set_index("code")["category"]

    forward = {}
    for code in frame.index:
        future = returns.loc[asof:, code].iloc[1:13].dropna()
        forward[code] = (1 + future).prod() - 1 if len(future) >= 10 else np.nan
    frame["forward_1y"] = pd.Series(forward)
    frame = frame.dropna()
    frame["target"] = frame.groupby("category")["forward_1y"].rank(pct=True)
    frame["asof"] = asof
    return frame

year_ends = [
    date for date in returns.index
    if date.month == 12 and date.year >= 2020
    and date <= returns.index.max() - pd.DateOffset(months=12)
]
panel = pd.concat([features_at(date) for date in year_ends])

features = ["cagr", "vol", "sharpe", "maxdd", "consistency", "momentum", "age"]
X = pd.get_dummies(panel[features + ["category"]], columns=["category"], drop_first=True).astype(float)
train_mask = (panel["asof"] < year_ends[-1]).to_numpy()
test_mask = ~train_mask

scaler = StandardScaler().fit(X.iloc[train_mask])
model = Ridge(alpha=1.0).fit(
    scaler.transform(X.iloc[train_mask]),
    panel["target"].to_numpy()[train_mask],
)
prediction = model.predict(scaler.transform(X.iloc[test_mask]))
ic = spearmanr(prediction, panel["target"].to_numpy()[test_mask]).statistic
use_ml = bool(ic > 0.05 and test_mask.sum() >= 20)
ml_weight = 0.20 if use_ml else 0.0
print(f"Forward-1y rank IC: {ic:.3f}  (test funds: {test_mask.sum()})")
print(f"ML blend weight: {ml_weight:.0%}" if use_ml else "ML gate failed -> keeping the rule-based score")

if use_ml:
    latest = X.iloc[(panel["asof"] == year_ends[-1]).to_numpy()]
    latest_rank = pd.Series(
        model.predict(scaler.transform(latest)), index=latest.index
    ).rank(pct=True) * 100
    metrics["ml_score"] = metrics["code"].map(latest_rank)
    metrics["score"] = (1 - ml_weight) * metrics["score"] + ml_weight * metrics["ml_score"].fillna(metrics["score"])
    metrics = metrics.sort_values("score", ascending=False).reset_index(drop=True)

Forward-1y rank IC: -0.320  (test funds: 162)
ML gate failed -> keeping the rule-based score


In [11]:
picked = [metrics[metrics["category"] == cat].head(BASE_QUOTA) for cat in CATEGORIES]
top = pd.concat(picked)

for _, row in metrics.drop(top.index).iterrows():
    if len(top) >= TOP_N:
        break
    top = pd.concat([top, row.to_frame().T])

top = top.sort_values("score", ascending=False).reset_index(drop=True)
top["rank"] = top.index + 1

print(f"Top {len(top)} selected")
print(top["category"].value_counts())
top[["rank", "name", "category", "score", "cagr_3y", "sharpe_3y", "max_drawdown", "sip_xirr"]].head(12)

Top 200 selected
category
index    68
flexi    39
large    32
mid      31
small    30
Name: count, dtype: int64


,rank,name,category,score,cagr_3y,sharpe_3y,max_drawdown,sip_xirr
0,1,WhiteOak Capital Mid Cap Fund - Direct Plan - ...,mid,96.703,0.208,0.817,-0.193,0.176
1,2,BANK OF INDIA FLEXI CAP FUND - Direct Plan - G...,flexi,92.154,0.190,0.668,-0.237,0.159
2,3,ITI Flexi Cap Fund - Direct Plan - Growth Option,flexi,90.712,0.184,0.683,-0.214,0.132
3,4,HSBC Midcap Fund - Direct Plan - Growth,mid,90.108,0.219,0.744,-0.260,0.181
4,5,BANDHAN Small Cap Fund - Direct Plan - Growth,small,88.845,0.235,0.829,-0.243,0.216
5,6,Invesco India Large Cap Fund - Direct Plan - G...,large,88.197,0.139,0.455,-0.369,0.106
6,7,WhiteOak Capital Large Cap Fund - Direct Plan ...,large,86.979,0.131,0.465,-0.152,0.074
7,8,360 ONE Flexicap Fund - Direct Plan - GROWTH O...,flexi,86.040,0.176,0.706,-0.181,0.097
8,9,Invesco India Mid Cap Fund - Direct Plan - Growth,mid,85.871,0.222,0.780,-0.341,0.186
9,10,Taurus Large Cap Fund - Direct Plan - Growth,large,85.817,0.144,0.533,-0.351,0.111


## 6. Build a recommendation

Change the five inputs below, then run the cells from here onward.

- **Age** and **horizon** set risk capacity.
- **Risk tolerance** is 1 (very low) to 5 (very high).
- **MODE** is `"sip"` or `"lumpsum"`.
- **AMOUNT** is rupees per month for a SIP, or a one-time amount for lump sum.

The engine first decides **how much should be in equity**. Because this universe has no debt funds, the remaining share is shown as an explicit **debt/cash gap**.

In [12]:
AGE = 30
HORIZON = 15          # years
MODE = "sip"          # "sip" or "lumpsum"
RISK = 4              # 1 = very low, 5 = very high
AMOUNT = 10_000       # rupees

def equity_target(age, horizon, risk):
    '''How much of the money should be in equity, in percent.'''
    h = np.clip((horizon - 1) / 14, 0, 1)
    a = np.clip((65 - age) / 45, 0, 1)
    r = (risk - 1) / 4
    capacity = 0.45 * h + 0.30 * a + 0.25 * r
    target = 20 + 65 * capacity

    horizon_cap = 10 if horizon < 1 else 30 if horizon < 3 else 55 if horizon < 5 else 75 if horizon < 7 else 90
    age_cap = max(20, 110 - age)
    risk_cap = {1: 30, 2: 50, 3: 70, 4: 85, 5: 95}[risk]
    return float(np.clip(min(target, horizon_cap, age_cap, risk_cap), 10, 95))

EQUITY = equity_target(AGE, HORIZON, RISK)
print(f"Age {AGE}, horizon {HORIZON}y, risk {RISK}/5, mode {MODE}")
print(f"Target equity: {EQUITY:.0f}%  |  debt/cash gap: {100 - EQUITY:.0f}%")

Age 30, horizon 15y, risk 4/5, mode sip
Target equity: 77%  |  debt/cash gap: 23%


The equity sleeve is split across categories. Short horizons hard-block mid and small caps; longer horizons allow them. The three profiles are starting points, and the horizon gates override them.

In [13]:
PROFILES = {
    "conservative": {"large": 0.40, "index": 0.25, "flexi": 0.25, "mid": 0.10, "small": 0.00},
    "moderate":     {"large": 0.25, "index": 0.20, "flexi": 0.30, "mid": 0.18, "small": 0.07},
    "aggressive":   {"large": 0.15, "index": 0.10, "flexi": 0.25, "mid": 0.28, "small": 0.22},
}

def category_targets(age, horizon, risk):
    equity = equity_target(age, horizon, risk) / 100
    profile = "conservative" if equity < 0.45 else "moderate" if equity < 0.70 else "aggressive"
    target = pd.Series(PROFILES[profile], dtype=float)

    # Hard horizon gates.
    if horizon < 3:
        target[["mid", "small"]] = 0
    elif horizon < 5:
        target["small"] = 0
        target["mid"] = min(target["mid"], 0.10)
    elif horizon < 7:
        target["small"] = min(target["small"], 0.05)
        target["mid"] = min(target["mid"], 0.20)

    return target / target.sum()

CATEGORY = category_targets(AGE, HORIZON, RISK)
print("Equity sleeve weights:")
print((CATEGORY * 100).round(1).to_string())

Equity sleeve weights:
large   15.000
index   10.000
flexi   25.000
mid     28.000
small   22.000


### Pick funds and optimise weights

Up to two funds are shortlisted per category. SIP mode tilts toward funds with strong SIP XIRR; lump-sum mode tilts toward Calmar (return per unit of drawdown).

A constrained mean-variance optimiser then chooses final weights:

- fully invested, long-only, 3–30% per fund,
- each category within ±5 percentage points of its target,
- portfolio volatility below a risk-based budget,
- objective: expected return minus risk penalty.

If the optimiser fails, the notebook falls back to the category targets split across the shortlisted funds.

In [14]:
pool = top.copy()
for cat in CATEGORIES:
    in_cat = pool["category"] == cat
    if MODE == "sip":
        pool.loc[in_cat, "pick_score"] = (
            0.7 * pool.loc[in_cat, "score"]
            + 0.3 * pool.loc[in_cat, "sip_xirr"].rank(pct=True) * 100
        )
    else:
        pool.loc[in_cat, "pick_score"] = (
            0.7 * pool.loc[in_cat, "score"]
            + 0.3 * pool.loc[in_cat, "calmar_3y"].rank(pct=True) * 100
        )

chosen = []
for cat, weight in CATEGORY.items():
    if weight <= 0:
        continue
    candidates = pool[(pool["category"] == cat) & (pool["years"] >= 2)]
    if candidates.empty:
        candidates = pool[pool["category"] == cat]
    candidates = candidates.sort_values("pick_score", ascending=False)
    chosen.append(candidates.head(2 if weight >= 0.15 else 1))
chosen = pd.concat(chosen).drop_duplicates("code").set_index("code")

history = returns[chosen.index].iloc[-36:].fillna(0)
cov = LedoitWolf().fit(history).covariance_ * 12

mu = (
    0.5 * chosen["cagr_3y"].fillna(chosen["cagr_1y"])
    + 0.5 * chosen.groupby("category")["cagr_3y"].transform("median")
).fillna(0.08).clip(-0.05, 0.25)

risk_aversion = {1: 8, 2: 6, 3: 4.5, 4: 3.5, 5: 2.5}[RISK]
vol_budget = {1: 0.14, 2: 0.15, 3: 0.17, 4: 0.19, 5: 0.21}[RISK]

cat_bounds = {}
for cat in CATEGORIES:
    target = CATEGORY.get(cat, 0)
    cat_bounds[cat] = (
        0.0 if target == 0 else max(0, target - 0.05),
        0.0 if target == 0 else min(1, target + 0.05),
    )

constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
for cat in CATEGORIES:
    mask = (chosen["category"] == cat).to_numpy().astype(float)
    low, high = cat_bounds[cat]
    constraints.append({"type": "ineq", "fun": lambda w, m=mask, h=high: h - w @ m})
    constraints.append({"type": "ineq", "fun": lambda w, m=mask, l=low: w @ m - l})
constraints.append({"type": "ineq", "fun": lambda w, b=vol_budget: b - np.sqrt(w @ cov @ w)})

x0 = np.array([CATEGORY[chosen.loc[code, "category"]] for code in chosen.index])
x0 = x0 / x0.sum()

target_weights = x0.copy()

def objective(w):
    expected = w @ mu.to_numpy()
    risk = w @ cov @ w
    drift = np.sum((w - target_weights) ** 2)
    return -(expected - 0.5 * risk_aversion * risk - 1.0 * drift)

result = minimize(
    objective, x0, method="SLSQP",
    bounds=[(0.03, 0.30)] * len(chosen),
    constraints=constraints,
)
if not result.success:
    # Retry without the volatility cap if it was too tight.
    result = minimize(
        objective, x0, method="SLSQP",
        bounds=[(0.03, 0.30)] * len(chosen),
        constraints=constraints[:-1],
    )

weights = pd.Series(result.x if result.success else x0, index=chosen.index)
weights[weights < 0.02] = 0
weights = weights / weights.sum()
print("Optimizer:", "solved" if result.success else "fallback used")
print(f"Expected volatility: {np.sqrt(weights.to_numpy() @ cov @ weights.to_numpy()):.1%}")

Optimizer: solved
Expected volatility: 17.3%


### Portfolio results

We check the portfolio on historical data, replay two stress periods, and run a block bootstrap Monte Carlo for the chosen horizon.

In [15]:
portfolio_returns = (returns[weights.index] * weights).sum(axis=1).dropna()
recent = portfolio_returns[
    portfolio_returns.index >= portfolio_returns.index.max() - pd.DateOffset(years=5)
]
benchmark_returns = market.reindex(recent.index).fillna(0)

def performance(r):
    curve = (1 + r).cumprod()
    years = len(r) / 12
    cagr = curve.iloc[-1] ** (1 / years) - 1
    vol = r.std() * np.sqrt(12)
    drawdown = (curve / curve.cummax() - 1).min()
    return {
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe": (cagr - RISK_FREE) / vol if vol > 0 else np.nan,
        "Max drawdown": drawdown,
    }

stats = pd.DataFrame({
    "Portfolio": performance(recent),
    "Market (Nifty 500)": performance(benchmark_returns),
})
print(stats.round(3).to_string())
print("Portfolio SIP XIRR:", f"{sip_xirr((1 + portfolio_returns).cumprod()):.2%}")

              Portfolio  Market (Nifty 500)
CAGR              0.162               0.085
Volatility        0.154               0.147
Sharpe            0.633               0.140
Max drawdown     -0.206              -0.181
Portfolio SIP XIRR: 17.59%


In [16]:
rng = np.random.default_rng(42)
months = max(1, int(HORIZON * 12))
paths = 2000
draws = rng.choice(recent.to_numpy(), size=(paths, months), replace=True)

if MODE == "sip":
    value = np.zeros(paths)
    for month in range(months):
        value = value * (1 + draws[:, month]) + AMOUNT
    invested = AMOUNT * months
else:
    value = AMOUNT * np.cumprod(1 + draws, axis=1)[:, -1]
    invested = AMOUNT

curve = np.cumprod(1 + draws, axis=1)
curve = np.column_stack([np.ones(paths), curve])
peak = np.maximum.accumulate(curve, axis=1)
drawdowns = (curve / peak - 1).min(axis=1)

mc = pd.DataFrame({"value": value})
print(mc["value"].describe(percentiles=[0.05, 0.5, 0.95]).round(0).to_string())
print(f"Probability of ending below contributions: {(value < invested).mean():.1%}")
print(f"Median max drawdown: {np.median(drawdowns):.1%}")
print(f"Worst 5% max drawdown: {np.percentile(drawdowns, 5):.1%}")

for label, start, end in [("COVID 2020", "2020-02-01", "2020-04-30"),
                          ("2021-22", "2021-10-01", "2022-06-30")]:
    episode = portfolio_returns.loc[start:end]
    if len(episode):
        print(f"{label}: {(1 + episode).prod() - 1:.1%}")

count        2,000.000
mean     7,607,376.000
std      3,413,400.000
min      1,581,357.000
5%       3,439,328.000
50%      6,884,307.000
95%     13,981,874.000
max     30,134,008.000
Probability of ending below contributions: 0.1%
Median max drawdown: -22.3%
Worst 5% max drawdown: -35.8%
COVID 2020: -3.7%
2021-22: -6.5%


### Final recommendation

`amount` is per month for a SIP, or one-time for lump sum. The equity share comes from the suitability policy; the rest is explicitly left for debt/cash outside this universe.

In [17]:
recommendation = chosen.loc[weights[weights > 0].index].copy()
recommendation["weight"] = weights[weights > 0]
recommendation["equity_share"] = recommendation["weight"] * EQUITY / 100
recommendation["amount"] = recommendation["equity_share"] * AMOUNT
recommendation = recommendation.sort_values("weight", ascending=False)

print(f"Recommended equity portfolio ({MODE}, rupees {AMOUNT:,})")
recommendation[["name", "category", "weight", "amount", "score", "cagr_3y", "sharpe_3y", "max_drawdown"]]

Recommended equity portfolio (sip, rupees 10,000)


,name,category,weight,amount,score,cagr_3y,sharpe_3y,max_drawdown
code,,,,,,,,
150584,WhiteOak Capital Mid Cap Fund - Direct Plan - ...,mid,0.159,"1,219.624",96.703,0.208,0.817,-0.193
151036,HSBC Midcap Fund - Direct Plan - Growth,mid,0.154,"1,176.952",90.108,0.219,0.744,-0.260
151379,ITI Flexi Cap Fund - Direct Plan - Growth Option,flexi,0.131,"1,001.229",90.712,0.184,0.683,-0.214
148404,BANK OF INDIA FLEXI CAP FUND - Direct Plan - G...,flexi,0.129,991.989,92.154,0.190,0.668,-0.237
147946,BANDHAN Small Cap Fund - Direct Plan - Growth,small,0.128,982.678,88.845,0.235,0.829,-0.243
145678,BANK OF INDIA SMALL CAP FUND - Direct Plan - G...,small,0.118,904.189,85.400,0.206,0.638,-0.324
118870,Taurus Large Cap Fund - Direct Plan - Growth,large,0.069,527.540,85.817,0.144,0.533,-0.351
120392,Invesco India Large Cap Fund - Direct Plan - G...,large,0.062,473.195,88.197,0.139,0.455,-0.369
149894,Axis Nifty Smallcap 50 Index Fund - Direct Pla...,index,0.050,383.021,84.556,0.175,0.488,-0.277


In [18]:
deployment = pd.DataFrame({
    "bucket": ["Equity funds", "Debt / cash (outside universe)"],
    "share_%": [EQUITY, 100 - EQUITY],
    "amount": [AMOUNT * EQUITY / 100, AMOUNT * (100 - EQUITY) / 100],
})
deployment

,bucket,share_%,amount
0,Equity funds,76.604,"7,660.417"
1,Debt / cash (outside universe),23.396,"2,339.583"


In [19]:
fig = px.bar(
    recommendation.reset_index(), x="weight", y="name", color="category",
    orientation="h", title="Recommended weights",
)
fig.update_layout(yaxis_title="", xaxis_tickformat=".0%", height=420)
fig.show()

curve = pd.DataFrame({
    "Portfolio": (1 + recent).cumprod(),
    "Nifty 500": (1 + benchmark_returns).cumprod(),
})
curve.index.name = "date"
fig = px.line(curve.reset_index(), x="date", y=["Portfolio", "Nifty 500"],
              title="Value of 1 rupee invested in the portfolio")
fig.show()

fig = px.histogram(mc, x="value", nbins=50,
                   title=f"Monte Carlo after {HORIZON:g} years ({MODE})")
fig.add_vline(x=invested, line_dash="dash", annotation_text="contributions")
fig.show()

### Warnings and checks

The notebook does not hide suitability problems. It prints warnings when the horizon is short, the portfolio is concentrated, or a pick has a short history.

In [20]:
messages = []
if HORIZON < 3:
    messages.append("Horizon under 3 years: equity is unsuitable; most money belongs in debt/cash.")
elif HORIZON < 5 and EQUITY > 55:
    messages.append("Horizon under 5 years: equity allocation is at its upper limit.")
if (recommendation["years"] < 3).any():
    messages.append("Some picks have under 3 years of history; treat their metrics cautiously.")
if recommendation.groupby("amc")["weight"].sum().max() > 0.30:
    messages.append("One fund house exceeds 30% of the portfolio.")
if recommendation.loc[recommendation["category"].isin(["mid", "small"]), "weight"].sum() > 0.45:
    messages.append("Mid + small caps exceed 45% of the portfolio.")
if recommendation["name"].str.contains("smallcap|small cap", case=False, na=False).any():
    messages.append("The index sleeve includes a small-cap index fund; its risk is not captured by the category split.")
if not use_ml:
    messages.append("ML gate failed: the score is rule-based only.")
print("\n".join(f"- {message}" for message in messages) if messages else "No major warnings.")

final_category = recommendation.groupby("category")["weight"].sum()
print("\nFinal category weights:")
print((final_category * 100).round(1).to_string())

- Mid + small caps exceed 45% of the portfolio.
- The index sleeve includes a small-cap index fund; its risk is not captured by the category split.
- ML gate failed: the score is rule-based only.

Final category weights:
category
flexi   26.000
index    5.000
large   13.100
mid     31.300
small   24.600


In [21]:
assert abs(recommendation["weight"].sum() - 1) < 0.01, "weights must sum to 1"
assert set(recommendation.index).issubset(set(top["code"])), "picks must come from the top 200"
print("Basic checks passed")

grid = []
for risk in range(1, 6):
    for horizon in [1, 3, 5, 10, 20]:
        grid.append({
            "risk": risk,
            "horizon": horizon,
            "equity_%": round(equity_target(30, horizon, risk)),
        })
pd.DataFrame(grid).pivot(index="risk", columns="horizon", values="equity_%")

Basic checks passed


horizon,1,3,5,10,20
risk,,,,,
1,30,30,30,30,30
2,30,43,48,50,50
3,30,47,52,62,70
4,30,52,56,66,77
5,30,55,60,70,80


## 7. Customise it

Useful settings:

- `TOP_N`, `MIN_YEARS`, `BASE_QUOTA` in the setup cell.
- `WEIGHTS` to change what "quality" means.
- `AGE`, `HORIZON`, `MODE`, `RISK`, `AMOUNT` in the recommendation section.
- `PROFILES` and `category_targets` to change the sleeve mix.
- `vol_budget` and `risk_aversion` to change how aggressively the optimiser behaves.

To force a fresh download, delete `data/cache/master.parquet` and `data/cache/raw/`. To refresh only the computed tables, delete everything in `data/cache/` except `raw/`.

**Known limitations**

- No AUM, expense ratio, manager, or holdings data exists on mfapi.in.
- "Top 200" is a quality score, not a size ranking.
- Benchmarks are equal-weight index-fund composites, not official indices.
- Only live schemes are listed, so survivorship bias remains.
- Category labels are inferred from names; a few funds may be misclassified.
- Index funds are one category. A small-cap index fund still carries small-cap risk even though it sits in the index sleeve.
- Historical returns do not predict future returns.